# RAGAS Evaluation

### Environment
- **LLM (generation + RAGAS judge):** `gpt-4o-mini` via the OpenAI API
- **Retrieval:** FAISS
- **Retrieval embeddings:** `all-MiniLM-L6-v2` (local)
- **Evaluation embeddings:** `text-embedding-3-small`
- **Evaluation framework:** RAGAS

### Evaluation setup

Four metrics are used: faithfulness, answer relevancy, context precision, and context recall.

The notebook covers baseline evaluation, question-level score analysis, and a comparison with degraded retrieval.

## Step1: Install dependencies

In [ ]:
import sys
print(sys.version)

In [ ]:
#%pip install --force-reinstall --no-cache-dir numpy==1.26.4 pandas==2.2.2

In [ ]:
!pip install -q ragas==0.1.21 langchain-openai langchain datasets \
         sentence-transformers faiss-cpu openai
print("✅ Dependencies ready")

## Step2: OpenAI API key

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
print("✅ API key set")

## Step3: Imports & configuration

In [ ]:
import json, warnings
import numpy as np
import pandas as pd

# RAG pipeline
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI

# RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

warnings.filterwarnings("ignore")

# ── Model config ──────────────────────────────────────────────────────────────
OPENAI_MODEL  = "gpt-4o-mini"             # generation + RAGAS evaluation
EMBED_MODEL   = "all-MiniLM-L6-v2"        # RAG retrieval embeddings (local)
OPENAI_EMBED  = "text-embedding-3-small"  # RAGAS answer-relevancy embeddings

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment

print(f"LLM          : {OPENAI_MODEL}")
print(f"RAG embeds   : {EMBED_MODEL}")
print(f"RAGAS embeds : {OPENAI_EMBED}")
print("✅ Config done")

## Step4: Knowledge base

In [ ]:
CORPUS = [
    "Mercury is the smallest planet in the Solar System and the closest to the Sun. "
    "It has no moons and no significant atmosphere. A day on Mercury (one rotation) "
    "lasts about 59 Earth days, but a year (orbit around the Sun) only takes 88 Earth days.",

    "Mars has two small moons named Phobos and Deimos. The planet has the tallest volcano "
    "in the Solar System, Olympus Mons, which stands about 22 km high. Mars has a thin "
    "atmosphere composed mainly of carbon dioxide.",

    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. It observes "
    "the universe in infrared light and can see farther back in time than any previous "
    "telescope. JWST is located at the L2 Lagrange point, about 1.5 million km from Earth.",

    "Saturn is famous for its ring system, which consists mainly of ice particles and rocky "
    "debris. Saturn has 146 confirmed moons — the most of any planet in the Solar System. "
    "Its largest moon, Titan, has a thick nitrogen atmosphere and liquid methane lakes.",

    "The Apollo 11 mission landed the first humans on the Moon on July 20, 1969. Astronauts "
    "Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited "
    "above in the command module. They returned safely to Earth on July 24, 1969.",

    "The speed of light in a vacuum is approximately 299,792 km/s. Light from the Sun takes "
    "about 8 minutes and 20 seconds to reach Earth. The nearest star to the Sun, Proxima "
    "Centauri, is about 4.24 light-years away.",
]

print(f"📄 Corpus: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i}] {doc[:70]}…")

## Step5: RAG pipeline

In [ ]:
# ── Build FAISS index ────────────────────────────────────────────────────────
print("Building embedding index…")
embedder = SentenceTransformer(EMBED_MODEL)
doc_embeddings = embedder.encode(CORPUS, show_progress_bar=False)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
index.add(doc_embeddings / norms)  # normalise for cosine similarity

print(f"✅ Index built — {index.ntotal} vectors")

def retrieve(query: str, top_k: int = 2) -> list[str]:
    """Return top-k most relevant documents for a query."""
    q_emb = embedder.encode([query])
    q_emb = q_emb / np.linalg.norm(q_emb)
    _, ids = index.search(q_emb, top_k)
    return [CORPUS[i] for i in ids[0]]

def generate(question: str, contexts: list[str]) -> str:
    """Generate an answer given a question and retrieved context."""
    context_text = "\n\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(contexts))
    prompt = (
        f"Answer the question using ONLY the information in the context below.\n"
        f"If the context doesn't contain the answer, say 'I don't know.'\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

# Quick smoke test
test_ctx = retrieve("What is the tallest volcano in the Solar System?")
test_ans = generate("What is the tallest volcano in the Solar System?", test_ctx)
print(f"\n🧪 Smoke test answer:\n  {test_ans[:120]}")

## Step6: Create the evaluation dataset

In [ ]:
# ── Define questions with known ground truths ─────────────────────────────
QA_PAIRS = [
    {
        "question": "How many moons does Saturn have?",
        "ground_truth": "Saturn has 146 confirmed moons, the most of any planet.",
    },
    {
        "question": "When was the James Webb Space Telescope launched?",
        "ground_truth": "The James Webb Space Telescope was launched on December 25, 2021.",
    },
    {
        "question": "Who were the astronauts on Apollo 11 who walked on the Moon?",
        "ground_truth": "Neil Armstrong and Buzz Aldrin walked on the Moon during Apollo 11.",
    },
    {
        "question": "How long does light from the Sun take to reach Earth?",
        "ground_truth": "Light from the Sun takes about 8 minutes and 20 seconds to reach Earth.",
    },
    {
        "question": "What is the atmosphere of Mars primarily made of?",
        "ground_truth": "Mars has a thin atmosphere composed mainly of carbon dioxide.",
    },
]

# ── Run RAG to generate answers + retrieve contexts ───────────────────────
print("Running RAG pipeline for each question…")
questions, answers, contexts, ground_truths = [], [], [], []

for qa in QA_PAIRS:
    ctx  = retrieve(qa["question"], top_k=2)
    ans  = generate(qa["question"], ctx)

    questions.append(qa["question"])
    answers.append(ans)
    contexts.append(ctx)
    ground_truths.append(qa["ground_truth"])

    print(f"  Q: {qa['question'][:60]}")
    print(f"  A: {ans[:80]}\n")

print("✅ Evaluation dataset ready")

## Step7: Configure RAGAS with OpenAI

In [ ]:
# ── Wire up OpenAI → RAGAS ────────────────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model=OPENAI_MODEL, temperature=0)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model=OPENAI_EMBED)
)

print(f"✅ RAGAS LLM        → OpenAI {OPENAI_MODEL}")
print(f"✅ RAGAS Embeddings → OpenAI {OPENAI_EMBED}")

## Step8: Run RAGAS evaluation

This calls the LLM several times per example (once per metric that needs reasoning).

In [ ]:
# ── Build HuggingFace Dataset ─────────────────────────────────────────────
eval_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       answers,
    "contexts":     contexts,
    "ground_truth": ground_truths,
})

print(f"Dataset: {len(eval_dataset)} rows")
print(f"Columns: {eval_dataset.column_names}\n")

# ── Run evaluation ────────────────────────────────────────────────────────
print("⏳ Running RAGAS evaluation…")

result = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n✅ Evaluation complete!")
print(result)

## Step9: Results

In [ ]:
# ── Overall scores ───────────────────────────────────────────────────────
scores_df = result.to_pandas()

print("=" * 65)
print(f"{'Metric':<22} {'Score':>8}   {'Interpretation'}")
print("=" * 65)

metric_info = {
    "faithfulness":      ("1.0 = fully grounded in context",   "⚠️ < 0.7 = hallucination risk"),
    "answer_relevancy":  ("1.0 = perfectly on-topic",          "⚠️ < 0.7 = drifts from question"),
    "context_precision": ("1.0 = retrieved docs all relevant",  "⚠️ < 0.7 = too much noise"),
    "context_recall":    ("1.0 = retrieved everything needed",  "⚠️ < 0.7 = missing key info"),
}

for metric, (good, bad) in metric_info.items():
    if metric in scores_df.columns:
        avg = scores_df[metric].mean()
        flag = "✅" if avg >= 0.7 else "❌"
        hint = good if avg >= 0.7 else bad
        print(f"  {flag} {metric:<20} {avg:>6.3f}   {hint}")

print("=" * 65)
print()

# ── Per-question table ────────────────────────────────────────────────────
display_cols = ["question"] + [c for c in scores_df.columns if c in metric_info]
short_q = scores_df["question"].str[:45] + "…"
display_df = scores_df[display_cols].copy()
display_df["question"] = short_q
print(display_df.to_string(index=False, float_format="{:.3f}".format))

## Bonus: Find the weakest examples

In [ ]:
import warnings; warnings.filterwarnings("ignore")

metrics_to_check = [c for c in scores_df.columns if c in metric_info]
if not metrics_to_check:
    print("No metric columns found — rerun the evaluation cell.")
else:
    for metric in metrics_to_check:
        worst_idx = scores_df[metric].idxmin()
        row = scores_df.iloc[worst_idx]
        print(f"{'─'*65}")
        print(f"  Lowest {metric}: {row[metric]:.3f}")
        print(f"  Q: {row['question']}")
        print(f"  A: {answers[worst_idx][:120]}")
        print(f"  GT: {ground_truths[worst_idx]}")
        print()

## Experiment: intentionally degrade retrieval

What happens to RAGAS scores when we retrieve *wrong* documents on purpose? This shows how RAGAS catches retrieval failures.

In [ ]:
import random

print("Running RAG with random (wrong) context…")
bad_answers, bad_contexts = [], []

for qa in QA_PAIRS:
    # Pick 2 random documents (not guaranteed to be relevant)
    shuffled = random.sample(CORPUS, 2)
    ans  = generate(qa["question"], shuffled)
    bad_answers.append(ans)
    bad_contexts.append(shuffled)

bad_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       bad_answers,
    "contexts":     bad_contexts,
    "ground_truth": ground_truths,
})

print("⏳ Evaluating degraded RAG…")
bad_result = evaluate(
    bad_dataset,
    metrics=[faithfulness, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

bad_df = bad_result.to_pandas()
good_means = scores_df[metrics_to_check].mean()
bad_means  = bad_df[[c for c in bad_df.columns if c in metric_info]].mean()

print("\n📊 Good retrieval vs random retrieval:")
print(f"{'Metric':<25} {'Good':>8}  {'Random':>8}  {'Delta':>8}")
print("-" * 55)
for m in bad_means.index:
    if m in good_means.index:
        delta = bad_means[m] - good_means[m]
        arrow = "⬇️" if delta < -0.05 else ("⬆️" if delta > 0.05 else "≈")
        print(f"  {m:<23} {good_means[m]:>8.3f}  {bad_means[m]:>8.3f}  {delta:>+8.3f} {arrow}")